In [ ]:
import os
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1 as documentai
from typing import Dict, Any
from finance_analysis.config import global_config as glob   

def process_invoice_raw(
    project_id: str,
    location: str,
    processor_id: str,
    file_path: str,
    mime_type: str = "application/pdf",
) -> Dict[str, Any]:
    """
    Call Document AI (Invoice Parser or Enterprise OCR) and return the raw Document as a dict.
    """
    client_options = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)

    name = client.processor_path(project_id, location, processor_id)

    with open(file_path, "rb") as f:
        file_content = f.read()

    raw_document = documentai.RawDocument(content=file_content, mime_type=mime_type)

    request = documentai.ProcessRequest(
        name=name,
        raw_document=raw_document,
    )

    result = client.process_document(request=request)

    # This is the full raw protobuf object:
    proto_doc = result.document

    # Option 1: keep it as protobuf (you can inspect fields directly)
    # Option 2: convert to a Python dict (JSON-like)
    document_dict = documentai.Document.to_dict(proto_doc)

    return document_dict


In [ ]:
invoice_name = "Free2move_DE_251113_4248127_3285000079177057.pdf"
invoice_name = "Departure_taxi.pdf"
invoice_name = "Hotel_Invoice.pdf"

doc = process_invoice_raw(
    project_id="neme-ai-rnd-dev-prj-01",
    location="eu",
    processor_id="c48f6c912b9ff9d5",
    file_path=os.path.join(glob.DATA_PKG_DIR, invoice_name)
)

In [ ]:
doc

# Comparison: Docling vs Document AI

Let's compare two approaches for extracting invoice data:
1. **Docling** (current approach) - OCR + Layout parsing with Docling library
2. **Google Document AI** - Cloud-based document processing with Layout Parser

In [ ]:
# Test the same invoice with Docling (current approach)
from finance_analysis.resources.document_processor import DocumentProcessor

# Use the same invoice file
docling_processor = DocumentProcessor(
    file_path=os.path.join(glob.DATA_PKG_DIR, invoice_name)
)

# Process the document
docling_doc = docling_processor.process()

In [ ]:
docling_doc.export_to_markdown()

In [ ]:
# Display Docling extracted content as markdown
docling_processor.display_markdown()

## Document AI Output as Markdown

**Note:** Document AI Layout Parser returns raw block layout without semantic table structure (no header/data row distinction), resulting in messy tables. This demonstrates why Docling is better for invoice processing.

In [ ]:
from IPython.display import Markdown, display

def docai_to_markdown(doc_dict):
    """
    Convert Document AI blocks to clean markdown format.
    
    Processes text blocks and table blocks separately, formatting tables
    with proper headers and maintaining document structure.
    """
    lines, table_rows = [], []
    
    # Iterate through all layout blocks in the document
    for block in doc_dict.get('document_layout', {}).get('blocks', []):
        
        # Handle text blocks (paragraphs, headings)
        if 'text_block' in block and (text := block['text_block'].get('text', '').strip()):
            # Flush any accumulated table rows before adding text
            if table_rows:
                lines.extend(_format_table(table_rows))
                table_rows = []
            
            # Determine block type and add appropriate markdown prefix
            block_type = block['text_block'].get('type_', 'paragraph')
            prefix = '\n# ' if block_type == 'heading-1' else '\n## ' if block_type == 'heading-2' else ''
            lines.append(f"{prefix}{text}\n")
        
        # Handle table blocks - collect rows for batch processing
        elif 'table_block' in block:
            for row in block['table_block'].get('body_rows', []):
                # Extract text from each cell, joining multiple text blocks within a cell
                row_data = [' '.join(cb['text_block'].get('text', '').strip() 
                           for cb in cell.get('blocks', []) if 'text_block' in cb) or '-'
                           for cell in row.get('cells', [])]
                # Only keep rows that have at least one non-empty cell
                if any(r != '-' for r in row_data):
                    table_rows.append(row_data)
    
    # Flush any remaining table rows at the end
    if table_rows:
        lines.extend(_format_table(table_rows))
    
    return "\n".join(lines)

def _format_table(rows):
    """
    Format rows as a proper markdown table with auto-detected headers.
    
    Args:
        rows: List of lists containing cell data
        
    Returns:
        List of formatted markdown table lines
    """
    # Normalize all rows to have the same number of columns
    max_cols = max(len(r) for r in rows)
    rows = [r + ['-'] * (max_cols - len(r)) for r in rows]
    
    # Auto-detect header: use first row if it's mostly populated
    has_header = len(rows) > 1 and sum(bool(c != '-') for c in rows[0]) >= max_cols // 2
    header = rows[0] if has_header else [f"Col{i+1}" for i in range(max_cols)]
    data = rows[1:] if has_header else rows
    
    # Build markdown table with header, separator, and data rows
    return [
        "\n| " + " | ".join(header) + " |",           # Header row
        "| " + " | ".join(['---'] * max_cols) + " |",  # Separator
        *["| " + " | ".join(r) + " |" for r in data],  # Data rows
        ""  # Blank line after table
    ]

# Convert and display the Document AI output as formatted markdown
display(Markdown(docai_to_markdown(doc)))

Wrapped up:

In [ ]:
import os
from finance_analysis.config import global_config as glob   
from finance_analysis.resources.document_processor import DocumentProcessor_GCP

# invoice_name = "Free2move_DE_251113_4248127_3285000079177057.pdf"
# invoice_name = "Departure_taxi.pdf"
invoice_name = "Hotel_Invoice.pdf"

docs = DocumentProcessor_GCP(
    file_path=os.path.join(glob.DATA_PKG_DIR, invoice_name),
    project_id="neme-ai-rnd-dev-prj-01",
    location="eu",
    processor_id="c48f6c912b9ff9d5"
)
result = docs.process()

docs.to_markdown()

In [ ]:
docs.display_markdown()